ดูว่าจังหวะตัดตอนพูดต้องตัดตอนกี่วิ split chunk

overlap text ที่ merge กันจะเคลียยังไง

โมเดลรับไฟล์เสียงสั้นสุดเท่าไร โมเดลถูกจูนมาแล้วให้รับไฟล์เสียงสั้นได้

อัดเสียงยาวๆแล้วซอยไฟล์ให้มัน overlap กับแล้วเทส

ออกแบบจัดเก็บไฟล์เสียงเป็น temp file

lib สร้าง temp file

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from package.typhoon.inference import TyphoonASR

d:\Git\leonidas-hermes\transcribe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2025-11-20 18:20:50 nemo_logging:393] d:\Git\leonidas-hermes\transcribe\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


In [3]:
model = TyphoonASR()

🌪️ Loading Typhoon ASR model on CPU...
[NeMo I 2025-11-20 18:20:53 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 2048 tokens


[NeMo W 2025-11-20 18:20:54 nemo_logging:393] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/workspace/warit/nemo-asr/stt_th_conformer_transducer_large/prepare_data/typhoon_cleanser/20250814/Split_gg/train_data_typhoon_asr_realtime.jsonl
    sample_rate: 16000
    batch_size: 8
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 30.0
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    
[NeMo W 2025-11-20 18:20:54 nemo_logging:393] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation d

[NeMo I 2025-11-20 18:20:54 nemo_logging:381] PADDING: 0


[NeMo W 2025-11-20 18:20:54 nemo_logging:393] d:\Git\leonidas-hermes\transcribe\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
      warnings.warn(
    


[NeMo I 2025-11-20 18:20:55 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-11-20 18:20:55 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2025-11-20 18:20:55 nemo_logging:393] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2025-11-20 18:20:55 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2025-11-20 18:20:55 nemo_logging:393] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2025-11-20 18:20:55 nemo_logging:381] Model EncDecRNNTBPEModel was successfully restored from C:\Users\terjr\.cache\huggingface\hub\models--scb10x--typhoon-asr-realtime\snapshots\a14b79d50c788dbdfe559c8a28a9b90153cf3865\typhoon-asr-realtime.nemo.


In [9]:
import glob

# ถอดเสียงทีละ chunk
chunk_files = sorted(glob.glob("audio/recording_byte/*.bin"))
# chunk_files = sorted(glob.glob("audio/chunks_fah/*.wav"))

texts = []

for chunk_file in chunk_files:
    processed = model.preprocess(chunk_file)
    result = model.transcribe(processed)
    text = result['text'][0] if result['text'] else ''
    texts.append(text)

texts

✅ Saved: processed_audio.wav (2.0s)


Transcribing: 100%|██████████| 1/1 [00:00<00:00,  9.34it/s]


✅ Saved: processed_audio.wav (2.0s)


Transcribing: 100%|██████████| 1/1 [00:00<00:00, 12.58it/s]


✅ Saved: processed_audio.wav (2.0s)


Transcribing: 100%|██████████| 1/1 [00:00<00:00, 11.90it/s]


✅ Saved: processed_audio.wav (0.2s)


Transcribing: 100%|██████████| 1/1 [00:00<00:00, 19.41it/s]


['ทะโลเซ', 'นี่คือใบฝ่าย', 'เทสหนึ่งสองสามสี่', '']

In [6]:
from package.chunk.merge_overlap import merge_overlapped_text, merge_overlapped_text_fuzzy

# รวมข้อความ
final_text = merge_overlapped_text(texts)
print(f"\noutput: {final_text}")

final_text = merge_overlapped_text_fuzzy(texts=texts, threshold=85)
print(f"\noutput: {final_text}")


output: ทะโลเซนี่คือใบฝ่ายเทสหนึ่งสองสามสี่

output: ทะโลเซนี่คือใบฝ่ายเทสหนึ่งสองสามสี่


เทสว่า chunk กี่วิ overlap กี่วิอันไหนเร็วกว่ากัน ในตอนทำ realtime